# Business Continuity & Disaster Recovery (BCDR) in Snowflake
Complete beginner-friendly guide with concepts, terminology, and code examples.

*Co-authored with CoCo*

---
## 1. Foundational Concepts (Before Snowflake)

Before diving into Snowflake-specific features, you need to understand **why** BCDR exists.

### What is Availability?
**Availability** = the percentage of time a system is operational and accessible.

| Term | Meaning |
|------|--------|
| **Uptime** | System is working and serving requests |
| **Downtime** | System is unavailable (planned or unplanned) |
| **High Availability (HA)** | Design goal: minimize downtime (e.g., 99.99% uptime = ~52 min downtime/year) |

### What is Disaster Recovery (DR)?
**Disaster Recovery** = Process to restore systems after a catastrophic failure (region outage, data corruption, natural disaster).

### What is Business Continuity (BC)?
**Business Continuity** = Ability to keep business services running with minimal interruption during failures. DR is a *subset* of BC.

### Key Metrics You Must Know

| Metric | Full Name | Meaning | Example |
|--------|-----------|---------|--------|
| **RPO** | Recovery Point Objective | Maximum acceptable data loss measured in time | RPO = 30 min means you can tolerate losing up to 30 minutes of data |
| **RTO** | Recovery Time Objective | Maximum acceptable downtime before recovery | RTO = 1 hour means system must be back within 1 hour |

**Simple analogy:**
- RPO = "How much data can I afford to lose?" (backup frequency)
- RTO = "How long can I afford to be down?" (recovery speed)


There is no relationship between RPO and RTO. They are independent metrics that measure different things.

## 2. Core Terminology Explained

### Replication
**Replication** = copying data and objects from one Snowflake account (source) to another (target) on a schedule.

- Requires at least two Snowflake accounts:
    * Primary account
    * Secondary account
- Can be in the same region or a different region, and even across different cloud platforms (AWS, Azure, GCP), as long as Snowflake supports replication between them.
- The **source** account holds the **primary** objects (read-write).
- The **target** account(s) hold **secondary** objects (read-only replicas).
- Replication happens **asynchronously** — there's always a small lag.
- **Edition requirements:**
    * Database and share replication are available to **all editions**.
    * Replication of other account objects (roles, warehouses, users, etc.) & failover/failback require **Business Critical Edition** (or higher).

```
[Source Account - US West]          [Target Account - US East]
   PRIMARY objects     ──────►      SECONDARY objects
   (read-write)        replication   (read-only)
```

### Replication Account
A **replication account** is simply another Snowflake account within the same organization that participates in replication. It can be:
- A **source account** (has primary/original data)
- A **target account** (receives replicated copies)

All accounts must belong to the **same Snowflake organization**.

### Replication Group
A **replication group** is a defined collection of objects in a source account that are replicated as a unit to one or more target accounts. Replication groups provide **read-only** access for the replicated objects in the target account.

This is a three-step process:

a) Create a **primary replication group** in the source account and define the objects to be replicated.

b) Create a **secondary replication group** in the target account as a replica of the primary replication group.

c) Synchronize the secondary replication group:
   - If a replication schedule is configured, Snowflake automatically performs refreshes according to that schedule, including the initial synchronization.
   - Otherwise, execute a manual refresh on the secondary replication group in the target account.

**Objects that can be included in a replication group:**
- Databases
- Shares (inbound and outbound)
- Users
- Roles (and grants)
- Warehouses
- Resource monitors
- Integrations (security, API, notification, storage)
- Network policies
- Account parameters

### Failover
**Failover** = promoting a secondary (replica) to become the new primary when the original primary is unavailable.

- Requires at least two Snowflake accounts:
    * Primary account
    * Secondary account
- Can be in the same region or a different region, as long as Snowflake supports replication between them.
- **Requires Business Critical Edition (or higher).**
- Before failover: Target account has **read-only** copies
- After failover: Target account becomes **read-write** (the new primary)
- The original source account's copies become secondary (read-only)

```
BEFORE failover:
  Account A (primary, read-write) ──► Account B (secondary, read-only)

AFTER failover:
  Account A (secondary, read-only) ◄── Account B (NEW primary, read-write)
```

### Failover Group
A **failover group** is a replication group that can also fail over. It is a logical collection of objects that are replicated from a source account to a target account and can be **promoted (failed over)** to the target account during a DR event.

- A secondary failover group provides read-only access until it is promoted.
- When promoted, the secondary failover group becomes the primary (read-write), and the former primary becomes secondary.

This is a three-step process:

a) Create a **primary failover group** in the source account and define the objects to be replicated.

b) Create a **secondary failover group** in the target account as a replica of the primary failover group.

c) Synchronize the secondary failover group:
   - If a replication schedule is configured, Snowflake automatically performs refreshes according to the schedule, including the initial synchronization.
   - Otherwise, execute a manual refresh on the secondary failover group in the target account.

> **Key difference:** A replication group only provides read-only copies. A failover group provides read-only copies AND the ability to promote the secondary to primary.

### Failback
**Failback** = returning operations back to the original primary account after the outage is resolved.

It is essentially a "reverse failover":
1. The outage in the original region is resolved.
2. You refresh the original account with latest data from the current primary.
3. You promote the original account back to primary (by calling `ALTER FAILOVER GROUP ... PRIMARY` on the original account).
4. Redirect clients back to the original account.

```
During outage:   Account B is primary (serving traffic)
After resolved:  Account A is promoted back to primary (failback)
```

### Client Redirect
**Client Redirect** enables automatic redirection of client connections from a source account to a target account (e.g., during failover), using a **connection URL**.

- A **connection object** is used to define a redirect target.
- Clients connect using the connection URL instead of the account URL.
- When failover occurs, the connection URL can be pointed to the new primary account.
- This minimizes manual client reconfiguration during DR events.

### Backup (Newer replacement for Snapshot Set)
**Backup** in Snowflake refers to **backup sets** — containers that hold point-in-time backups of a database, schema, or table.

- A **Backup Set** is the container (created with `CREATE BACKUP SET`).
- Individual **Backups** are point-in-time captures added to a backup set (manually via `ALTER BACKUP SET` or automatically via a backup policy schedule).
- Backups are stored **within the same account**.
- Protects against accidental data **changes** or deletion.
- They complement Time Travel and replication (but are separate mechanisms).
- A **Backup Policy** can define: schedule (cron or interval), expiration period, and optional retention lock (for compliance — immutable backups).
- Available to **all Snowflake editions**. Retention locks and legal holds require **Business Critical Edition**.
- **Granularity**: You can create backup sets at table, schema, or database level.

### Snapshot Set (Deprecated)
**Snapshot Set** was the predecessor to Backup Set. It provided the same functionality (point-in-time backups within the same account). The `CREATE SNAPSHOT SET` command is now deprecated and replaced by `CREATE BACKUP SET`.

> **Note:** Do not confuse the deprecated Snapshot Set feature with the internal concept of a "snapshot" during replication. When Snowflake replicates data, it takes a consistent point-in-time snapshot of all objects in the group and transfers it to the target. This ensures **point-in-time consistency** — all replicated objects reflect the same moment in time. This is an internal mechanism, not the deprecated SNAPSHOT SET feature.

### Comparison Table

| Feature | Same Account | Different Account | Same Region | Different Region | Cross-Cloud |
|---------|:---:|:---:|:---:|:---:|:---:|
| **Backup Set** | Yes | No | Yes | No | No |
| **Time Travel** | Yes | No | Yes | No | No |
| **Replication Group** | No | Yes | Yes | Yes | Yes |
| **Failover Group** | No | Yes | Yes | Yes | Yes |

### RPO and RTO Considerations
- **RPO (Recovery Point Objective):** Determined by the replication schedule frequency. More frequent replication = less data loss.
- **RTO (Recovery Time Objective):** Determined by how quickly failover can be executed and clients redirected. Snowflake failover is typically fast (minutes), but client redirect configuration affects total RTO.

## Client Redirect in Snowflake

Client Redirect provides a **stable connection URL** that you can point to different accounts without changing application code. It is implemented through a **connection object**.

---

### What is a Connection Object?

A connection object is a **DNS routing mechanism**. It generates a stable URL that routes clients to whichever Snowflake account currently holds the **primary** connection — without embedding any credentials or account-specific details.

| Type | Format | Purpose |
|------|--------|----------|
| **Account URL** (direct) | `orgname-accountname.snowflakecomputing.com` | Connects to a specific account. No redirect capability. |
| **Connection URL** (redirectable) | `orgname-connection_name.snowflakecomputing.com` | Connects to whichever account holds the primary connection. Enables redirect. |

---

### How Client Redirect Works

1. You create a **connection object** → it generates a connection URL.
2. Clients connect using this **connection URL** (not the account URL).
3. The account holding the **primary connection** receives all traffic for that URL.
4. During failover, you promote the secondary connection in another account → clients are redirected transparently.

---
**Client connection string format:**
```
# Instead of connecting directly to an account:
host = 'myorg-myaccount.snowflakecomputing.com'

# Use the connection URL (redirectable):m
host = 'myorg-my_app_connection.snowflakecomputing.com'
```

All clients using the connection URL will automatically be routed to whichever account is currently the primary for that connection.

### Setup Process (4 Steps)

#### Step 1: Create a primary connection in the source account

This creates the connection object. It is automatically the **primary** connection and generates the connection URL.

```sql
-- Run in the SOURCE account (Account A)
CREATE CONNECTION my_app_connection;
```

#### Step 2: Enable failover to target accounts (REQUIRED)

This authorizes which accounts are **allowed to promote** the connection to primary. Without this step, the target account **cannot** take over the connection during failover.

```sql
-- Run in the SOURCE account (Account A)
-- Each target account MUST be in a DIFFERENT region than the source.
ALTER CONNECTION my_app_connection
  ENABLE FAILOVER TO ACCOUNTS myorg.account_b;

-- To allow multiple target accounts:
-- ALTER CONNECTION my_app_connection
--   ENABLE FAILOVER TO ACCOUNTS myorg.account_b, myorg.account_c;
```

> **Without this command, failover of the connection WILL NOT WORK.** This is analogous to `ALLOWED_ACCOUNTS` in `CREATE FAILOVER GROUP`.

#### Step 3: Create a secondary (replica) connection in the target account

This creates a replica of the primary connection in the target account. The secondary is **dormant** — it does not receive traffic until promoted.

```sql
-- Run in the TARGET account (Account B)
CREATE CONNECTION my_app_connection
  AS REPLICA OF myorg.account_a.my_app_connection;
```

> The secondary connection **must have the same name** as the primary. Snowflake enforces this.

#### Step 4: Promote to primary (during failover)

When the source account is unavailable, promote the secondary connection in the target account. All clients using the connection URL are now routed to the target account.

```sql
-- Run in the TARGET account (Account B) to take over traffic
ALTER CONNECTION my_app_connection PRIMARY;
```

> After promotion, the former primary connection automatically becomes secondary.

---

### Failback (Reverse the redirect)

Once the original account recovers, promote its connection back to primary:

```sql
-- Run in the ORIGINAL source account (Account A) to reclaim traffic
ALTER CONNECTION my_app_connection PRIMARY;
```

---

### Disabling Failover for a Connection

To revoke a target account's ability to promote the connection:

```sql
-- Run in the SOURCE account
ALTER CONNECTION my_app_connection
  DISABLE FAILOVER TO ACCOUNTS myorg.account_b;
```

---

### Using the Connection URL (Client Side)

The connection URL replaces the normal account URL in your client configuration. Authentication (user, password, role) is still provided by the client as usual.

**Python Connector:**
```python
import snowflake.connector

conn = snowflake.connector.connect(
    host='myorg-my_app_connection.snowflakecomputing.com',
    account='myorg-my_app_connection',
    user='MY_USER',
    password='...',
    warehouse='COMPUTE_WH',
    role='SYSADMIN'
)
```

**JDBC:**
```
jdbc:snowflake://myorg-my_app_connection.snowflakecomputing.com/?user=MY_USER&password=...&warehouse=COMPUTE_WH
```

---

### Viewing Connections

```sql
SHOW CONNECTIONS;
```

---

### All Connection Commands Summary

| Command | Where to Run | What It Does |
|---------|:---:|---|
| `CREATE CONNECTION <name>` | Source (primary) | Creates a primary connection object |
| `ALTER CONNECTION <name> ENABLE FAILOVER TO ACCOUNTS ...` | Source (primary) | Authorizes target accounts to promote (**required**) |
| `CREATE CONNECTION <name> AS REPLICA OF ...` | Target (secondary) | Creates a secondary (dormant) replica |
| `ALTER CONNECTION <name> PRIMARY` | Target (to promote) | Promotes secondary to primary (redirects traffic) |
| `ALTER CONNECTION <name> DISABLE FAILOVER TO ACCOUNTS ...` | Source (primary) | Revokes target account's promotion rights |
| `SHOW CONNECTIONS` | Any account | Lists all connections and their status |

---

### Key Points

| Aspect | Detail |
|--------|--------|
| **Edition Required** | Business Critical or higher |
| **Stores credentials?** | No — only a routing entry |
| **Multiple per account?** | Yes — create multiple connections for different workloads |
| **Primary constraint** | Only one account can hold the primary for a given connection name at a time |
| **Region constraint** | Target accounts in `ENABLE FAILOVER` must be in a different region than the source |
| **Name constraint** | Secondary must have the same name as the primary (enforced) |
| **Use with** | Failover Groups for full BCDR (connection handles traffic redirect, failover group handles data) |

---
## 3. Snowflake BCDR Architecture Overview

Snowflake's BCDR is built on two main features:

| Feature | Purpose | Edition Required |
|---------|---------|------------------|
| **Replication & Failover Groups** | Replicate objects + enable failover | Database/share replication: All editions. Full account replication + failover: Business Critical+ |
| **Client Redirect** | Redirect client connections to a different account | Business Critical+ |

### How They Work Together

```
┌─────────────────────────────────────────────────────────────────────┐
│                    NORMAL OPERATIONS                                 │
│                                                                     │
│  [Your Apps/BI Tools]                                               │
│         │                                                           │
│         ▼                                                           │
│  [Connection URL] ──────► [Account A - US-WEST-2]                   │
│                              │  PRIMARY failover group              │
│                              │  (databases, users, roles,           │
│                              │   warehouses, integrations)          │
│                              │                                      │
│                              │  replication (every 10 min)          │
│                              ▼                                      │
│                           [Account B - US-EAST-1]                    │
│                              SECONDARY failover group (read-only)    │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                    DURING OUTAGE                                     │
│                                                                     │
│  [Your Apps/BI Tools]                                               │
│         │                                                           │
│         ▼                                                           │
│  [Connection URL] ──────► [Account B - US-EAST-1]                   │
│         (redirected)         PRIMARY failover group (read-write)     │
│                                                                     │
│                           [Account A - US-WEST-2]                    │
│                              (unreachable / outage)                  │
└─────────────────────────────────────────────────────────────────────┘
```

---
## 4. What Can Be Added to Replication Group and Failover Group?

The `OBJECT_TYPES` parameter in `CREATE REPLICATION GROUP` / `CREATE FAILOVER GROUP` accepts the following values:

| Object Type | Replication Group | Failover Group |
|-------------|:-:|:-:|
| Databases | ✔ | ✔ |
| Shares (inbound & outbound) | ✔ | ✔ |
| Users | ✔ | ✔ |
| Roles (+ grants) | ✔ | ✔ |
| Warehouses | ✔ | ✔ |
| Integrations (security, API, notification, storage, external access) | ✔ | ✔ |
| Network Policies | ✔ | ✔ |
| Resource Monitors | ✔ | ✔ |
| Account Parameters | ✔ | ✔ |
| External Volumes | ✔ | ✔ |

### Important Notes

- You **cannot** add specific file_formats, schemas, stages, tables, or views to a replication/failover group individually. You add **databases** as a whole unit, and all objects within those databases are replicated.
- You **can** select which specific databases to include using the `ALLOWED_DATABASES` parameter (database-level granularity, not object-level).
- Similarly, `ALLOWED_SHARES` controls which shares are replicated, and `ALLOWED_INTEGRATION_TYPES` filters which integration types to include.
- **Connections** (for Client Redirect) are managed separately — they are not part of the group's `OBJECT_TYPES`. You create them independently in each account.

```sql
-- Example: creating a failover group with specific object types and databases
CREATE FAILOVER GROUP my_fg
  OBJECT_TYPES = DATABASES, ROLES, USERS, WAREHOUSES, INTEGRATIONS, NETWORK_POLICIES
  ALLOWED_DATABASES = my_db1, my_db2
  ALLOWED_INTEGRATION_TYPES = SECURITY_INTEGRATIONS
  ALLOWED_ACCOUNTS = myorg.target_account
  REPLICATION_SCHEDULE = '10 MINUTE';
```

## Useful SHOW Commands for Replication

These commands help you **discover and verify** your replication setup before creating replication/failover groups.

---

### SHOW REPLICATION ACCOUNTS

**Purpose:** Lists all accounts in your organization that are enabled for replication.

**When to use:**
- Before setting up replication, to confirm which target accounts are available.
- To verify that a target account exists in the desired region/cloud.
- To find the exact account name and region needed for `ALLOWED_ACCOUNTS` in `CREATE FAILOVER GROUP`.

**What it returns:** Account name, region, cloud platform, and whether the account is enabled for replication.

```sql
SHOW REPLICATION ACCOUNTS;
```

> **Example scenario:** You want to replicate to US-East. Run this command to confirm you have an account there before creating your failover group.

---

### SHOW REPLICATION DATABASES

**Purpose:** Lists all primary and secondary databases that have replication enabled in your account.

**When to use:**
- To check which databases are already being replicated (and their role — primary or secondary).
- To verify replication status after setting up a failover/replication group.
- To see the source account and region for each secondary database.
- To troubleshoot replication — confirm a database is actually included in replication.

**What it returns:** Database name, whether it's primary or secondary, source account, region, and replication group membership.

```sql
SHOW REPLICATION DATABASES;
```

> **Example scenario:** After creating a failover group, run this in the target account to confirm the secondary databases were created successfully.

---

### SHOW REGIONS

**Purpose:** Lists all Snowflake regions where accounts can be created.

**When to use:**
- During DR planning, to decide which region to place your secondary account in (geographic diversity).
- To find the correct region identifier needed when requesting a new account from Snowflake.
- To understand which cloud platforms and regions are available for cross-region/cross-cloud replication.

**What it returns:** Region name, cloud platform (AWS/Azure/GCP), and region group.

```sql
SHOW REGIONS;
```

> **Example scenario:** You're planning a DR strategy and need to pick a region on a different cloud platform. Run this to see all available options.

---

### Summary: When to Use Each

| Command | Question It Answers |
|---------|--------------------|
| `SHOW REPLICATION ACCOUNTS` | "Which accounts in my org can I replicate to?" |
| `SHOW REPLICATION DATABASES` | "Which databases are currently being replicated, and what's their status?" |
| `SHOW REGIONS` | "What regions/clouds are available for placing a new account?" |

## Naming Convention: Must Primary and Secondary Use the Same Name?

Yes — it's not just good practice, it's **required** (enforced by Snowflake).

When you create a secondary failover group, replication group, or connection object using `AS REPLICA OF`, Snowflake requires it to have the **same name** as the primary. You cannot choose a different name.

### Why?

| Object | Why same name is required |
|--------|---------------------------|
| **Failover Group** | The secondary is a replica of a specific named primary group. Snowflake tracks them as a pair by name. |
| **Replication Group** | Same as failover group — the secondary replicates a specific named group. |
| **Connection Object** | The connection URL is derived from the name (`org-connection_name.snowflakecomputing.com`). If names differed across accounts, the redirect mechanism wouldn't work. |

### Example

```sql
-- Source (Account A)
CREATE FAILOVER GROUP my_fg ...;
CREATE CONNECTION app_conn;

-- Target (Account B) — MUST use the same name
CREATE FAILOVER GROUP my_fg
  AS REPLICA OF myorg.account_a.my_fg;

CREATE CONNECTION app_conn
  AS REPLICA OF myorg.account_a.app_conn;
```

> You don't need to worry about naming conventions here — Snowflake won't let you use a different name for the secondary. The `AS REPLICA OF` syntax creates it with the same name automatically.

## End-to-End: Failover Group + Connection Object + Failback

This walkthrough covers the complete process across **two accounts**:
- **Account A** (source, `myorg.account_a`, US-West) — the original primary
- **Account B** (target, `myorg.account_b`, US-East) — the DR secondary

---

### Understanding SUSPEND and RESUME (Read This First)

`SUSPEND` and `RESUME` control **scheduled replication refreshes** on a secondary failover group.

| Command | What it does | Where to run it |
|---------|-------------|------------------|
| `ALTER FAILOVER GROUP <name> SUSPEND;` | Pauses automatic scheduled refreshes. The secondary **stops receiving updates** from the primary. | Only on a **secondary** account |
| `ALTER FAILOVER GROUP <name> RESUME;` | Resumes automatic scheduled refreshes. The secondary **starts receiving updates** again. | Only on a **secondary** account |

**Key rules:**
1. **SUSPEND and RESUME are local operations.** They only affect the failover group in the account where you run them. They have zero effect on the other account.
2. **They can only run on a secondary.** If you try to run SUSPEND or RESUME on a primary failover group, it will fail — the primary is the source of data, not a receiver.
3. **Once you promote a secondary to primary, the suspension becomes irrelevant.** The account is no longer receiving updates (it IS the source now), so there's nothing to suspend or resume.
4. **You do NOT need to RESUME after promoting.** Promotion changes the role entirely — the concept of "receiving refreshes" no longer applies to a primary.
5. **You DO need to RESUME on the new secondary** (the account that just became secondary after failover/failback), so it starts receiving replicated updates from the new primary.
6. **SUSPEND only blocks automatic (scheduled) refreshes.** Manual `ALTER FAILOVER GROUP ... REFRESH` still works even when suspended.

**Common confusion clarified:**

> *"After I SUSPEND and then promote to PRIMARY, do I need to RESUME?"*
>
> **No.** Once promoted, that account is primary — SUSPEND/RESUME don't apply to it anymore. You only need RESUME on whichever account is now the **secondary** (the one that needs to start receiving updates).

> *"Does SUSPEND affect both accounts at the same time?"*
>
> **No.** SUSPEND only affects the local failover group in the account where you run it. The other account is completely unaware.

---

### Phase 1: Setup in Source Account (Account A)

```sql
-- ============================================================
-- RUN IN ACCOUNT A (Source / Primary)
-- Role: ACCOUNTADMIN
-- ============================================================

-- Step 1: Create the failover group with desired objects and databases
CREATE FAILOVER GROUP my_failover_group
  OBJECT_TYPES = DATABASES, USERS, ROLES, WAREHOUSES, INTEGRATIONS, NETWORK_POLICIES, ACCOUNT_PARAMETERS
  ALLOWED_DATABASES = prod_db, analytics_db
  ALLOWED_INTEGRATION_TYPES = SECURITY_INTEGRATIONS
  ALLOWED_ACCOUNTS = myorg.account_b
  REPLICATION_SCHEDULE = '10 MINUTE';

-- Step 2: Create the connection object (automatically becomes primary)
CREATE CONNECTION app_connection;

-- Step 3: Enable failover of the connection to the target account
-- THIS IS REQUIRED — without it, Account B CANNOT promote the connection
-- during failover. This authorizes which accounts can take over.
ALTER CONNECTION app_connection
  ENABLE FAILOVER TO ACCOUNTS myorg.account_b;

-- Step 4: Verify setup
SHOW FAILOVER GROUPS;
SHOW CONNECTIONS;
```

**What each parameter/command means:**

| Parameter / Command | Explanation |
|---------------------|-------------|
| `OBJECT_TYPES` | Categories of account objects to replicate (DATABASES, USERS, ROLES, etc.) |
| `ALLOWED_DATABASES` | Which specific databases to include in replication |
| `ALLOWED_INTEGRATION_TYPES` | Which integration types to replicate (e.g., SECURITY_INTEGRATIONS) |
| `ALLOWED_ACCOUNTS` | Target account(s) permitted to receive failover group replicas |
| `REPLICATION_SCHEDULE` | How often to auto-sync (e.g., every 10 minutes) |
| `ENABLE FAILOVER TO ACCOUNTS` | Authorizes which accounts can promote the connection to primary (**required for connection failover**) |

> **Why is `ENABLE FAILOVER TO ACCOUNTS` needed?**
> - `ALLOWED_ACCOUNTS` in `CREATE FAILOVER GROUP` authorizes replication of the **failover group**.
> - `ENABLE FAILOVER TO ACCOUNTS` on the **connection** authorizes promotion of the **connection**.
> - These are two separate authorization mechanisms for two separate objects.
> - Without `ENABLE FAILOVER TO ACCOUNTS`, the target can promote the failover group (data becomes writable), but **cannot** redirect client traffic via the connection.

---

### Phase 2: Setup in Target Account (Account B)

```sql
-- ============================================================
-- RUN IN ACCOUNT B (Target / Secondary)
-- Role: ACCOUNTADMIN
-- ============================================================

-- Step 1: Create the secondary failover group as a replica of the primary
CREATE FAILOVER GROUP my_failover_group
  AS REPLICA OF myorg.account_a.my_failover_group;

-- Step 2: Create the secondary connection (replica of the primary connection)
CREATE CONNECTION app_connection
  AS REPLICA OF myorg.account_a.app_connection;

-- Step 3: Verify — secondary objects should appear as read-only replicas
SHOW FAILOVER GROUPS;
SHOW CONNECTIONS;
SHOW REPLICATION DATABASES;
```

> At this point:
> - Account B is receiving replicated data every 10 minutes (per the schedule).
> - The connection URL (`myorg-app_connection.snowflakecomputing.com`) routes all clients to Account A.
> - Account B is authorized to promote both the failover group AND the connection.

---

### Phase 3: Client Configuration

Clients connect using the **connection URL** (not the account URL). This enables transparent redirect during failover — no application code changes needed.

```python
# Python connector — use the connection URL
import snowflake.connector

conn = snowflake.connector.connect(
    host='myorg-app_connection.snowflakecomputing.com',
    account='myorg-app_connection',
    user='MY_USER',
    password='...',
    warehouse='COMPUTE_WH'
)
```

---

### Phase 4: Failover (Disaster Strikes — Account A is Down)

When Account A becomes unavailable, promote Account B to primary.

```sql
-- ============================================================
-- RUN IN ACCOUNT B (Currently secondary, promoting to primary)
-- Role: ACCOUNTADMIN
-- ============================================================

-- Step 1: SUSPEND — stop receiving refreshes from Account A
-- WHY: If a scheduled refresh is in-flight, promotion will fail.
--      SUSPEND ensures no refresh is running so promotion succeeds.
ALTER FAILOVER GROUP my_failover_group SUSPEND;

-- Step 2: Promote the failover group — Account B becomes read-write (PRIMARY)
-- After this, B is primary. The SUSPEND above is now irrelevant
-- because B is no longer a secondary receiving updates.
ALTER FAILOVER GROUP my_failover_group PRIMARY;

-- Step 3: Promote the connection — client traffic redirects to Account B
-- This works because Account A previously ran:
--   ALTER CONNECTION app_connection ENABLE FAILOVER TO ACCOUNTS myorg.account_b;
ALTER CONNECTION app_connection PRIMARY;

-- Step 4: Verify promotion
SHOW FAILOVER GROUPS;   -- Should show Account B as primary
SHOW CONNECTIONS;        -- Should show app_connection as primary here

-- NOTE: No RESUME needed here. Account B is now PRIMARY.
-- SUSPEND/RESUME only apply to secondaries.
```

> **What happened to Account A?**
> Account A's failover group automatically became secondary. Once A comes back online, it will start receiving scheduled refreshes from B (the new primary) automatically.

---

### Phase 5: Failback (Account A Recovers — Return to Original)

Once Account A is back online, reverse the process to restore the original topology.

```sql
-- ============================================================
-- RUN IN ACCOUNT A (Currently secondary, will become primary again)
-- Role: ACCOUNTADMIN
-- ============================================================

-- Step 1: Refresh to get the latest data from Account B (current primary)
-- This ensures minimal data loss before switching back.
-- (Manual REFRESH works even if scheduled replication is suspended.)
ALTER FAILOVER GROUP my_failover_group REFRESH;

-- Step 2: SUSPEND — stop receiving refreshes from Account B
-- Same reason as before: prevent in-flight refresh from blocking promotion.
ALTER FAILOVER GROUP my_failover_group SUSPEND;

-- Step 3: Promote Account A's failover group back to primary
-- After this, A is primary again. B automatically becomes secondary.
ALTER FAILOVER GROUP my_failover_group PRIMARY;

-- Step 4: Promote Account A's connection back to primary (redirect clients back)
ALTER CONNECTION app_connection PRIMARY;

-- Step 5: Verify everything is back to normal
SHOW FAILOVER GROUPS;       -- Account A is primary again
SHOW CONNECTIONS;            -- app_connection primary is in Account A
SHOW REPLICATION DATABASES;  -- Databases are primary in Account A
```

```sql
-- ============================================================
-- RUN IN ACCOUNT B (Now secondary again — needs to start receiving updates)
-- Role: ACCOUNTADMIN
-- ============================================================

-- Step 6: RESUME — start receiving scheduled refreshes from Account A again
-- WHY: B is now secondary. We suspended B's replication back in Phase 4.
--      Since B was promoted and then demoted back to secondary, its
--      replication schedule may be in a suspended state. RESUME re-activates it.
ALTER FAILOVER GROUP my_failover_group RESUME;
```

> **After failback:** Account A is primary (read-write), Account B is secondary (read-only). Clients are routed back to Account A. Account B resumes receiving scheduled updates.

---

### Summary: Order of Operations

| Step | Failover (A → B) | Failback (B → A) |
|:----:|-------------------|-------------------|
| 1 | SUSPEND on B (stop receiving) | REFRESH on A (get latest from B) |
| 2 | Promote failover group on B | SUSPEND on A (stop receiving) |
| 3 | Promote connection on B | Promote failover group on A |
| 4 | Verify | Promote connection on A |
| 5 | — | Verify on A |
| 6 | — | RESUME on B (start receiving again) |

---

### The Golden Rule

```
SUSPEND → Promote Failover Group → Promote Connection → (on new secondary) RESUME
```

- **SUSPEND** prevents in-flight refresh conflicts (run on the secondary BEFORE promotion).
- **Promote failover group** makes data read-write in the new primary.
- **Promote connection** redirects client traffic to the new primary.
- **RESUME** re-activates scheduled replication on the new secondary (run AFTER the other account is promoted).

> If you redirect traffic (connection) before data is writable (failover group), clients will get errors. Always promote the failover group first, then the connection.

---

### Two Authorization Mechanisms (Don't Confuse Them)

| What you're authorizing | Command | Where to run | Without it... |
|---|---|:---:|---|
| Failover group replication to target | `CREATE FAILOVER GROUP ... ALLOWED_ACCOUNTS = myorg.account_b` | Source (primary) | Target cannot create a secondary failover group |
| Connection promotion by target | `ALTER CONNECTION ... ENABLE FAILOVER TO ACCOUNTS myorg.account_b` | Source (primary) | Target cannot promote the connection (traffic redirect fails) |

**After failover:**
- The target account now has **read-write** access to all objects in the group
- The original source account's objects become **secondary** (read-only)
- Applications connected via Client Redirect are pointed to the new primary

---
## 9. Backups (Backup Sets & Backup Policies)

Snowflake **Backups** provide immutable, point-in-time copies of your data for disaster recovery.

### How Backups Differ from Time Travel & Replication

| Feature | Purpose | Scope | Protection Against |
|---------|---------|-------|-------------------|
| **Time Travel** | Query/restore historical data | Single account, up to 90 days | Accidental changes (UPDATE/DELETE/DROP) |
| **Replication** | Copy data to another region/account | Cross-account, cross-region | Region outages |
| **Backups** | Immutable point-in-time snapshots | Single or cross-account | Accidental loss, compliance, ransomware |

### Key Concepts

| Term | Meaning |
|------|--------|
| **Backup Set** | An object that contains a series of backups for a specific database, schema, or table |
| **Backup Policy** | Controls the schedule and retention of automatic backups |
| **Retention Lock** | Makes backups truly immutable (cannot be deleted even by admins) — Business Critical+ |
| **Legal Hold** | Prevents backup deletion for legal/compliance reasons — Business Critical+ |

In [ ]:
%%sql -r create_backup_policy
-- Create a backup policy (controls schedule and retention)
CREATE BACKUP POLICY my_backup_policy
  BACKUP_FREQUENCY = '1 DAY'
  RETENTION_DAYS = 30;

In [ ]:
%%sql -r manage_backups
-- Create a backup set for a database and apply the policy
CREATE BACKUP SET my_db_backup
  FOR DATABASE my_production_db
  BACKUP_POLICY = my_backup_policy;

-- Manually trigger a backup
ALTER BACKUP SET my_db_backup CREATE BACKUP;

-- List available backups
SHOW BACKUPS IN BACKUP SET my_db_backup;

In [ ]:
%%sql -r restore_backup
-- Restore from a backup (creates a new database from the backup)
CREATE DATABASE my_restored_db
  FROM BACKUP OF my_production_db
  AT BACKUP_ID = '<backup_id_from_show_backups>';

---
## 11. Monitoring Replication

In [ ]:
%%sql -r replication_history
-- Check replication status and lag
SELECT
  REPLICATION_GROUP_NAME,
  PHASE_NAME,
  START_TIME,
  END_TIME,
  BYTES_TRANSFERRED,
  OBJECT_COUNT
FROM TABLE(INFORMATION_SCHEMA.REPLICATION_GROUP_REFRESH_HISTORY(
  GROUP_NAME => 'MY_FAILOVER_GROUP'
))
ORDER BY START_TIME DESC
LIMIT 10;

In [ ]:
%%sql -r replication_status
-- Check the overall status of failover groups
SHOW FAILOVER GROUPS;

-- Check replication lag (time since last successful refresh)
SELECT
  PRIMARY_ID,
  REPLICATION_GROUP_NAME,
  LAST_REFRESHED_AT,
  CURRENT_TIMESTAMP() AS NOW,
  TIMESTAMPDIFF('minute', LAST_REFRESHED_AT, CURRENT_TIMESTAMP()) AS LAG_MINUTES
FROM TABLE(INFORMATION_SCHEMA.REPLICATION_GROUP_REFRESH_PROGRESS(
  GROUP_NAME => 'MY_FAILOVER_GROUP'
));

---
## 14. Quick Reference — All Terms in One Place

| Term | One-Line Definition |
|------|--------------------|
| **Availability** | % of time the system is operational |
| **Disaster Recovery (DR)** | Plan to restore systems after a catastrophic failure |
| **Business Continuity (BC)** | Strategy to keep business running during/after disaster |
| **RPO** | Max acceptable data loss (in time) |
| **RTO** | Max acceptable downtime (in time) |
| **Replication** | Copying objects from source to target account on a schedule |
| **Replication Account** | Any Snowflake account participating in replication (source or target) |
| **Replication Group** | A set of objects replicated together (read-only at target) |
| **Failover Group** | A replication group that can also be promoted to primary (read-write) |
| **Primary** | The source/active copy (read-write) |
| **Secondary** | The replica copy (read-only until promoted) |
| **Failover** | Promoting a secondary to become the new primary |
| **Failback** | Returning primary status to the original account after outage resolves |
| **Snapshot** | Point-in-time consistent capture of all objects during replication |
| **Client Redirect** | Connection URL that can be pointed to different accounts |
| **Connection URL** | The redirectable endpoint clients use to connect |
| **Backup Set** | Object containing point-in-time backups of a database/schema/table |
| **Backup Policy** | Rules for automatic backup schedule and retention |
| **Retention Lock** | Makes backups immutable (can't be deleted) |
| **Time Travel** | Ability to query/restore past states within retention period |
| **Point-in-time consistency** | All objects in a snapshot reflect the exact same moment |

---
## 15. Edition Requirements Summary

| Feature | Standard | Enterprise | Business Critical | VPS |
|---------|:--------:|:----------:|:-----------------:|:---:|
| Database replication | ✔ | ✔ | ✔ | ✔ |
| Share replication | ✔ | ✔ | ✔ | ✔ |
| Full account object replication | ✘ | ✘ | ✔ | ✔ |
| Failover groups | ✘ | ✘ | ✔ | ✔ |
| Client Redirect | ✘ | ✘ | ✔ | ✔ |
| Backup sets (basic) | ✔ | ✔ | ✔ | ✔ |
| Retention lock / Legal hold | ✘ | ✘ | ✔ | ✔ |